In [7]:
from pathlib import Path
import pandas as pd
# Correct project root
PROJECT_ROOT = Path(
    "/Users/irfanhussain/Desktop/systemic-risk-project"
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
FFIEC_DIR = RAW_DIR / "ffiec"
FDIC_DIR = RAW_DIR / "fdic"

print("Project root:", PROJECT_ROOT)
print("FFIEC directory exists:", FFIEC_DIR.exists())
print("FDIC directory exists:", FDIC_DIR.exists())

Project root: /Users/irfanhussain/Desktop/systemic-risk-project
FFIEC directory exists: True
FDIC directory exists: True


In [8]:
fdic_files = [
    p for p in FDIC_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == ".csv"
]

print("FDIC CSV files:")
for p in fdic_files:
    print(p)

FDIC CSV files:
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/fdic/fdic_failures.csv


In [9]:
fdic_path = fdic_files[0]

fdic = pd.read_csv(
    fdic_path,
    low_memory=False
)

print("Shape:", fdic.shape)
display(fdic.head())

Shape: (21, 13)


,CERT,CHCLASS1,CITYST,COST,FAILDATE,FIN,ID,NAME,QBFASSET,QBFDEP,RESTYPE,RESTYPE1,SAVR
0,10716,NM,"COOPER, TX",18263.231,5/31/2019,10531,4097,ENLOE STATE BANK,36738,31254,FAILURE,PI,DIF
1,58112,NM,"LOUISA, KY",4264.465,10/25/2019,10532,4098,LOUISA COMMUNITY BANK,28163,25174,FAILURE,PA,DIF
2,58317,SB,"MAUMEE, OH",1570.592,10/25/2019,10533,4099,RESOLUTE BANK,23292,22885,FAILURE,PA,DIF
3,21111,N,"NEWARK, NJ",1226.619,11/1/2019,10534,4100,CITY NATIONAL BANK OF NEW JERSEY,120574,111234,FAILURE,PA,DIF
4,18265,NM,"ERICSON, NE",23278.878,2/14/2020,10535,4101,ERICSON STATE BANK,100879,95159,FAILURE,PA,DIF


In [10]:
print("Columns:")
print(fdic.columns.tolist())

print("\nFailure date range:")
print(
    pd.to_datetime(
        fdic["FAILDATE"],
        errors="coerce"
    ).min(),
    "→",
    pd.to_datetime(
        fdic["FAILDATE"],
        errors="coerce"
    ).max()
)

print("\nFailure transaction types:")
print(fdic["RESTYPE"].value_counts(dropna=False))

print("\nResolution types:")
print(fdic["RESTYPE1"].value_counts(dropna=False))

print("\nMissing CERT:")
print(fdic["CERT"].isna().sum())

print("\nDuplicate CERT:")
print(fdic["CERT"].duplicated().sum())

Columns:
['CERT', 'CHCLASS1', 'CITYST', 'COST', 'FAILDATE', 'FIN', 'ID', 'NAME', 'QBFASSET', 'QBFDEP', 'RESTYPE', 'RESTYPE1', 'SAVR']

Failure date range:
2019-05-31 00:00:00 → 2026-07-17 00:00:00

Failure transaction types:
RESTYPE
FAILURE    21
Name: count, dtype: int64

Resolution types:
RESTYPE1
PA    17
PI     4
Name: count, dtype: int64

Missing CERT:
0

Duplicate CERT:
0


In [11]:
fdic_clean = fdic.copy()

fdic_clean["cert"] = pd.to_numeric(
    fdic_clean["CERT"],
    errors="coerce"
)

fdic_clean["failure_date"] = pd.to_datetime(
    fdic_clean["FAILDATE"],
    errors="coerce"
)

print(fdic_clean[["CERT", "cert", "failure_date"]].head())

    CERT   cert failure_date
0  10716  10716   2019-05-31
1  58112  58112   2019-10-25
2  58317  58317   2019-10-25
3  21111  21111   2019-11-01
4  18265  18265   2020-02-14


In [12]:
# ============================================================
# REBUILD BANK-QUARTER PANEL
# ============================================================

years = sorted(
    int(p.name)
    for p in FFIEC_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

print("FFIEC years found:", years)

processed_years = []

for year in years:
    processed_years.append(
        process_ffiec_year(year)
    )

bank_quarter_panel = pd.concat(
    processed_years,
    ignore_index=True
)

bank_quarter_panel = (
    bank_quarter_panel
    .sort_values(
        ["IDRSSD", "Reporting Period End Date"]
    )
    .reset_index(drop=True)
)

print("\nBank-quarter panel rebuilt.")
print("Shape:", bank_quarter_panel.shape)

print(
    "Date range:",
    bank_quarter_panel["Reporting Period End Date"].min(),
    "to",
    bank_quarter_panel["Reporting Period End Date"].max()
)

FFIEC years found: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


NameError: name 'process_ffiec_year' is not defined

In [48]:
# ============================================================
# REBUILD FFIEC IDENTITY TABLE
# ============================================================

ffiec_identity = bank_quarter_panel[
    [
        "IDRSSD",
        "FDIC Certificate Number",
        "Financial Institution Name",
        "Financial Institution State",
        "Reporting Period End Date"
    ]
].copy()

ffiec_identity["IDRSSD"] = (
    ffiec_identity["IDRSSD"]
    .astype("string")
    .str.strip()
)

ffiec_identity["cert"] = normalize_cert(
    ffiec_identity["FDIC Certificate Number"]
)

print("ffiec_identity shape:", ffiec_identity.shape)

ffiec_identity shape: (157921, 6)


In [49]:
# ============================================================
# CHECK AMBIGUOUS CERTIFICATE NUMBERS
# ============================================================

cert_id_counts = (
    ffiec_identity
    .dropna(subset=["cert", "IDRSSD"])
    .groupby("cert")["IDRSSD"]
    .nunique()
)

problem_certs = cert_id_counts[
    cert_id_counts > 1
].index

print(
    "Certificates linked to multiple IDRSSDs:",
    len(problem_certs)
)

print("Problem certificates:")
print(list(problem_certs))

Certificates linked to multiple IDRSSDs: 6
Problem certificates:
[np.int64(1213), np.int64(8110), np.int64(15118), np.int64(20776), np.int64(32541), np.int64(57083)]


In [50]:
# ============================================================
# LOAD FDIC FAILURE DATA
# ============================================================

fdic_files = list(FDIC_DIR.glob("*.csv"))

print("FDIC CSV files found:")
for f in fdic_files:
    print(" ", f.name)

if not fdic_files:
    raise FileNotFoundError(
        f"No CSV files found in {FDIC_DIR}"
    )

fdic_path = fdic_files[0]

fdic_clean = pd.read_csv(fdic_path)

fdic_clean["cert"] = normalize_cert(
    fdic_clean["CERT"]
)

fdic_clean["failure_date"] = pd.to_datetime(
    fdic_clean["FAILDATE"],
    errors="coerce"
)

failure_problem_certs = (
    set(fdic_clean["cert"].dropna())
    & set(problem_certs)
)

print("\nFDIC failure records:", len(fdic_clean))
print(
    "Ambiguous certificates appearing in failures:",
    len(failure_problem_certs)
)
print(failure_problem_certs)

FDIC CSV files found:
  fdic_failures.csv

FDIC failure records: 21
Ambiguous certificates appearing in failures: 0
set()


In [51]:
# ============================================================
# FDIC FAILURE → FFIEC MATCHING
# ============================================================

fdic_events = fdic_clean[
    [
        "CERT",
        "NAME",
        "failure_date",
        "RESTYPE",
        "RESTYPE1"
    ]
].copy()

fdic_events = fdic_events.reset_index(drop=True)

# Unique identifier for each failure event
fdic_events["event_id"] = fdic_events.index

# Normalize certificate number
fdic_events["cert"] = normalize_cert(
    fdic_events["CERT"]
)

# Make sure dates are datetime
fdic_events["failure_date"] = pd.to_datetime(
    fdic_events["failure_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# Match every FDIC failure to FFIEC observations
# using certificate number
# ------------------------------------------------------------

failure_candidates = fdic_events.merge(
    ffiec_identity,
    on="cert",
    how="left"
)

# Only keep FFIEC observations occurring
# on or before the actual failure
failure_candidates = failure_candidates[
    failure_candidates["Reporting Period End Date"].notna()
    & failure_candidates["failure_date"].notna()
    & (
        failure_candidates["Reporting Period End Date"]
        <= failure_candidates["failure_date"]
    )
].copy()

# ------------------------------------------------------------
# Select the LAST FFIEC observation before failure
# for each failure event
# ------------------------------------------------------------

failure_matches = (
    failure_candidates
    .sort_values(
        ["event_id", "Reporting Period End Date"]
    )
    .groupby("event_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Total FDIC failures:", len(fdic_events))
print(
    "Matched to FFIEC:",
    failure_matches["IDRSSD"].notna().sum()
)
print(
    "Unmatched:",
    failure_matches["IDRSSD"].isna().sum()
)

Total FDIC failures: 21
Matched to FFIEC: 21
Unmatched: 0


In [52]:
failure_matches[
    [
        "event_id",
        "CERT",
        "NAME",
        "IDRSSD",
        "Financial Institution Name",
        "Financial Institution State",
        "Reporting Period End Date",
        "failure_date"
    ]
].sort_values("failure_date")

,event_id,CERT,NAME,IDRSSD,Financial Institution Name,Financial Institution State,Reporting Period End Date,failure_date
0,0,10716,ENLOE STATE BANK,431958,"ENLOE STATE BANK, THE",TX,2019-03-31,2019-05-31
1,1,58112,LOUISA COMMUNITY BANK,3462449,LOUISA COMMUNITY BANK,KY,2019-09-30,2019-10-25
2,2,58317,RESOLUTE BANK,3461264,RESOLUTE BANK,OH,2019-09-30,2019-10-25
3,3,21111,CITY NATIONAL BANK OF NEW JERSEY,567905,CITY NATIONAL BANK OF NEW JERSEY,NJ,2019-09-30,2019-11-01
4,4,18265,ERICSON STATE BANK,145956,ERICSON STATE BANK,NE,2019-12-31,2020-02-14
5,5,14361,THE FIRST STATE BANK,627425,"FIRST STATE BANK, THE",WV,2020-03-31,2020-04-03
6,6,16748,FIRST CITY BANK OF FLORIDA,695134,FIRST CITY BANK OF FLORIDA,FL,2020-09-30,2020-10-16
7,7,15426,ALMENA STATE BANK,224552,ALMENA STATE BANK,KS,2020-09-30,2020-10-23
8,8,24735,SILICON VALLEY BANK,802866,SILICON VALLEY BANK,CA,2022-12-31,2023-03-10
9,9,57053,SIGNATURE BANK,2942690,SIGNATURE BANK,NY,2022-12-31,2023-03-12


In [53]:
# ============================================================
# CLEAN FAILURE EVENTS TABLE
# ============================================================

failure_events = failure_matches[
    [
        "event_id",
        "IDRSSD",
        "cert",
        "CERT",
        "NAME",
        "failure_date",
        "RESTYPE",
        "RESTYPE1",
        "Reporting Period End Date"
    ]
].copy()

failure_events = failure_events.rename(
    columns={
        "Reporting Period End Date":
            "last_ffiec_observation_before_failure"
    }
)

failure_events = failure_events.sort_values(
    "failure_date"
).reset_index(drop=True)

print("Failure events shape:", failure_events.shape)

display(failure_events)

Failure events shape: (21, 9)


,event_id,IDRSSD,cert,CERT,NAME,failure_date,RESTYPE,RESTYPE1,last_ffiec_observation_before_failure
0,0,431958,10716,10716,ENLOE STATE BANK,2019-05-31,FAILURE,PI,2019-03-31
1,1,3462449,58112,58112,LOUISA COMMUNITY BANK,2019-10-25,FAILURE,PA,2019-09-30
2,2,3461264,58317,58317,RESOLUTE BANK,2019-10-25,FAILURE,PA,2019-09-30
3,3,567905,21111,21111,CITY NATIONAL BANK OF NEW JERSEY,2019-11-01,FAILURE,PA,2019-09-30
4,4,145956,18265,18265,ERICSON STATE BANK,2020-02-14,FAILURE,PA,2019-12-31
5,5,627425,14361,14361,THE FIRST STATE BANK,2020-04-03,FAILURE,PA,2020-03-31
6,6,695134,16748,16748,FIRST CITY BANK OF FLORIDA,2020-10-16,FAILURE,PA,2020-09-30
7,7,224552,15426,15426,ALMENA STATE BANK,2020-10-23,FAILURE,PA,2020-09-30
8,8,802866,24735,24735,SILICON VALLEY BANK,2023-03-10,FAILURE,PA,2022-12-31
9,9,2942690,57053,57053,SIGNATURE BANK,2023-03-12,FAILURE,PA,2022-12-31


In [54]:
print(
    failure_events[
        "last_ffiec_observation_before_failure"
    ].isna().sum(),
    "failures have no FFIEC observation"
)

0 failures have no FFIEC observation


In [55]:
# ============================================================
# CREATE FAILURE-WITHIN-NEXT-4-QUARTERS TARGET
# ============================================================

# One failure date per IDRSSD
failure_lookup = (
    failure_events[
        ["IDRSSD", "failure_date"]
    ]
    .dropna(subset=["IDRSSD", "failure_date"])
    .groupby("IDRSSD", as_index=False)
    .agg(
        failure_date=("failure_date", "min")
    )
)

# ------------------------------------------------------------
# Start from complete bank-quarter panel
# ------------------------------------------------------------

model_data = bank_quarter_panel.copy()

model_data["IDRSSD"] = (
    model_data["IDRSSD"]
    .astype("string")
    .str.strip()
)

model_data["Reporting Period End Date"] = pd.to_datetime(
    model_data["Reporting Period End Date"],
    errors="coerce"
)

# ------------------------------------------------------------
# Attach failure date to every bank-quarter observation
# ------------------------------------------------------------

model_data = model_data.merge(
    failure_lookup,
    on="IDRSSD",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Define one-year prediction horizon
# ------------------------------------------------------------

model_data["prediction_end"] = (
    model_data["Reporting Period End Date"]
    + pd.DateOffset(years=1)
)

# ------------------------------------------------------------
# Latest failure date in our FDIC dataset
# ------------------------------------------------------------

latest_failure_date = pd.to_datetime(
    failure_events["failure_date"]
).max()

print("Latest FDIC failure:", latest_failure_date)

# ------------------------------------------------------------
# Can we observe the full future year?
# ------------------------------------------------------------

model_data["complete_4q_horizon"] = (
    model_data["prediction_end"]
    <= latest_failure_date
)

# ------------------------------------------------------------
# Failure occurs AFTER observation and
# WITHIN next 4 quarters
# ------------------------------------------------------------

future_failure = (
    model_data["failure_date"].notna()
    & (
        model_data["failure_date"]
        > model_data["Reporting Period End Date"]
    )
    & (
        model_data["failure_date"]
        <= model_data["prediction_end"]
    )
)

# Initialize target as missing
model_data["failure_next_4q"] = pd.NA

# Only assign labels when full horizon is observable
valid_horizon = model_data["complete_4q_horizon"]

model_data.loc[
    valid_horizon,
    "failure_next_4q"
] = (
    future_failure.loc[valid_horizon]
    .astype("int8")
)

# ------------------------------------------------------------
# Remove observations occurring on/after failure
# ------------------------------------------------------------

model_data["after_failure"] = (
    model_data["failure_date"].notna()
    & (
        model_data["Reporting Period End Date"]
        >= model_data["failure_date"]
    )
)

# ------------------------------------------------------------
# Final ML dataset
# ------------------------------------------------------------

ml_data = model_data[
    model_data["complete_4q_horizon"]
    & ~model_data["after_failure"]
].copy()

ml_data["failure_next_4q"] = (
    ml_data["failure_next_4q"]
    .astype("int8")
)

print("\nModel dataset shape:", ml_data.shape)

print("\nTarget distribution:")
print(
    ml_data["failure_next_4q"]
    .value_counts()
)

print("\nTarget proportion:")
print(
    ml_data["failure_next_4q"]
    .value_counts(normalize=True)
)

Latest FDIC failure: 2026-07-17 00:00:00

Model dataset shape: (149092, 30)

Target distribution:
failure_next_4q
0    149024
1        68
Name: count, dtype: int64

Target proportion:
failure_next_4q
0    0.999544
1    0.000456
Name: proportion, dtype: float64


In [56]:
positive_cases = (
    ml_data[
        ml_data["failure_next_4q"] == 1
    ]
    .sort_values(
        ["IDRSSD", "Reporting Period End Date"]
    )
)

print(
    "Positive bank-quarter observations:",
    len(positive_cases)
)

print(
    "Unique banks with a future failure:",
    positive_cases["IDRSSD"].nunique()
)

display(
    positive_cases[
        [
            "IDRSSD",
            "Financial Institution Name",
            "Reporting Period End Date",
            "failure_date",
            "failure_next_4q"
        ]
    ].head(20)
)

Positive bank-quarter observations: 68
Unique banks with a future failure: 19


,IDRSSD,Financial Institution Name,Reporting Period End Date,failure_date,failure_next_4q
6815,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2023-06-30,2024-04-26,1
6816,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2023-09-30,2024-04-26,1
6817,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2023-12-31,2024-04-26,1
11514,145956,ERICSON STATE BANK,2019-03-31,2020-02-14,1
11515,145956,ERICSON STATE BANK,2019-06-30,2020-02-14,1
11516,145956,ERICSON STATE BANK,2019-09-30,2020-02-14,1
11517,145956,ERICSON STATE BANK,2019-12-31,2020-02-14,1
23520,224552,ALMENA STATE BANK,2019-12-31,2020-10-23,1
23521,224552,ALMENA STATE BANK,2020-03-31,2020-10-23,1
23522,224552,ALMENA STATE BANK,2020-06-30,2020-10-23,1


In [57]:
if len(positive_cases) > 0:

    example_bank = positive_cases.iloc[0]["IDRSSD"]

    example_history = (
        ml_data[
            ml_data["IDRSSD"] == example_bank
        ]
        .sort_values("Reporting Period End Date")
    )

    display(
        example_history[
            [
                "IDRSSD",
                "Financial Institution Name",
                "Reporting Period End Date",
                "capital_ratio",
                "loan_to_asset",
                "deposit_to_asset",
                "nonaccrual_to_asset",
                "nonaccrual_to_loan",
                "roa",
                "nii_to_asset",
                "failure_date",
                "failure_next_4q"
            ]
        ]
    )

,IDRSSD,Financial Institution Name,Reporting Period End Date,capital_ratio,loan_to_asset,deposit_to_asset,nonaccrual_to_asset,nonaccrual_to_loan,roa,nii_to_asset,failure_date,failure_next_4q
6794,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2018-03-31,0.084505,0.507457,0.87392,0.005733,0.011297,0.000918,0.007439,2024-04-26,0
6795,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2018-06-30,0.082239,0.517432,0.850274,0.005539,0.010705,0.001129,0.007446,2024-04-26,0
6796,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2018-09-30,0.079108,0.520215,0.917315,0.005086,0.009776,0.00103,0.007357,2024-04-26,0
6797,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2018-12-31,0.080449,0.523275,0.881842,0.003767,0.007198,0.001646,0.007408,2024-04-26,0
6798,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2019-03-31,0.080051,0.528116,0.896263,0.003194,0.006047,0.000453,0.006923,2024-04-26,0
6799,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2019-06-30,0.084249,0.514452,0.8646,0.003179,0.006179,0.000418,0.006847,2024-04-26,0
6800,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2019-09-30,0.080214,0.509969,0.89263,0.003874,0.007597,-0.000296,0.00637,2024-04-26,0
6801,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2019-12-31,0.073566,0.524596,0.902043,0.003725,0.007100,-0.000563,0.006055,2024-04-26,0
6802,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2020-03-31,0.075407,0.571818,0.896536,0.00431,0.007537,0.000028,0.006354,2024-04-26,0
6803,1216321,REPUBLIC FIRST BANK DBA REPUBLIC BANK,2020-06-30,0.056821,0.574474,0.824799,0.002874,0.005002,0.000722,0.005181,2024-04-26,0


In [58]:
# ============================================================
# MODELING SETUP
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

feature_cols = [
    "capital_ratio",
    "loan_to_asset",
    "deposit_to_asset",
    "nonaccrual_to_asset",
    "nonaccrual_to_loan",
    "roa",
    "nii_to_asset"
]

target_col = "failure_next_4q"

# Make sure chronological order is correct
ml_data = (
    ml_data
    .sort_values("Reporting Period End Date")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# TIME-BASED SPLIT
# ------------------------------------------------------------

train_end = pd.Timestamp("2023-12-31")
val_end = pd.Timestamp("2024-12-31")

train_data = ml_data[
    ml_data["Reporting Period End Date"] <= train_end
].copy()

val_data = ml_data[
    (ml_data["Reporting Period End Date"] > train_end)
    & (ml_data["Reporting Period End Date"] <= val_end)
].copy()

test_data = ml_data[
    ml_data["Reporting Period End Date"] > val_end
].copy()

# ------------------------------------------------------------
# X and y
# ------------------------------------------------------------

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_val = val_data[feature_cols]
y_val = val_data[target_col]

X_test = test_data[feature_cols]
y_test = test_data[target_col]

print("TRAIN")
print(train_data.shape)
print(y_train.value_counts())
print()

print("VALIDATION")
print(val_data.shape)
print(y_val.value_counts())
print()

print("TEST")
print(test_data.shape)
print(y_test.value_counts())
print()

print(
    "Train period:",
    train_data["Reporting Period End Date"].min(),
    "to",
    train_data["Reporting Period End Date"].max()
)

print(
    "Validation period:",
    val_data["Reporting Period End Date"].min(),
    "to",
    val_data["Reporting Period End Date"].max()
)

print(
    "Test period:",
    test_data["Reporting Period End Date"].min(),
    "to",
    test_data["Reporting Period End Date"].max()
)

TRAIN
(121764, 30)
failure_next_4q
0    121709
1        55
Name: count, dtype: int64

VALIDATION
(18332, 30)
failure_next_4q
0    18323
1        9
Name: count, dtype: int64

TEST
(8996, 30)
failure_next_4q
0    8992
1       4
Name: count, dtype: int64

Train period: 2018-03-31 00:00:00 to 2023-12-31 00:00:00
Validation period: 2024-03-31 00:00:00 to 2024-12-31 00:00:00
Test period: 2025-03-31 00:00:00 to 2025-06-30 00:00:00


In [59]:
# ============================================================
# LOGISTIC REGRESSION
# ============================================================

logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )
    )
])

logistic_pipeline.fit(X_train, y_train)

log_val_prob = logistic_pipeline.predict_proba(X_val)[:, 1]
log_test_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print(
    "Validation ROC-AUC:",
    roc_auc_score(y_val, log_val_prob)
)

print(
    "Validation PR-AUC:",
    average_precision_score(y_val, log_val_prob)
)

print(
    "Test ROC-AUC:",
    roc_auc_score(y_test, log_test_prob)
)

print(
    "Test PR-AUC:",
    average_precision_score(y_test, log_test_prob)
)

Logistic Regression
Validation ROC-AUC: 0.603982850940227
Validation PR-AUC: 0.0006383227987076557
Test ROC-AUC: 0.9592415480427046
Test PR-AUC: 0.09891358641358641


In [60]:
# ============================================================
# RANDOM FOREST
# ============================================================

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_pipeline.fit(X_train, y_train)

rf_val_prob = rf_pipeline.predict_proba(X_val)[:, 1]
rf_test_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest")
print(
    "Validation ROC-AUC:",
    roc_auc_score(y_val, rf_val_prob)
)

print(
    "Validation PR-AUC:",
    average_precision_score(y_val, rf_val_prob)
)

print(
    "Test ROC-AUC:",
    roc_auc_score(y_test, rf_test_prob)
)

print(
    "Test PR-AUC:",
    average_precision_score(y_test, rf_test_prob)
)

Random Forest
Validation ROC-AUC: 0.641658631834913
Validation PR-AUC: 0.0008888304569592185
Test ROC-AUC: 0.9703486432384342
Test PR-AUC: 0.17656517019999074


In [61]:
rf_model = rf_pipeline.named_steps["model"]

rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
})

rf_importance = (
    rf_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(rf_importance)

,feature,importance
0,capital_ratio,0.361441
1,nonaccrual_to_loan,0.154102
2,roa,0.140557
3,nii_to_asset,0.104095
4,nonaccrual_to_asset,0.096155
5,loan_to_asset,0.074614
6,deposit_to_asset,0.069037


In [62]:
try:
    import xgboost as xgb
    print("XGBoost installed:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost installed: 3.4.1


In [63]:
# ============================================================
# XGBOOST
# ============================================================

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

scale_pos_weight = negative_count / positive_count

print("Scale positive weight:", scale_pos_weight)

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        xgb.XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    )
])

xgb_pipeline.fit(X_train, y_train)

xgb_val_prob = xgb_pipeline.predict_proba(X_val)[:, 1]
xgb_test_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

print("\nXGBoost")

print(
    "Validation ROC-AUC:",
    roc_auc_score(y_val, xgb_val_prob)
)

print(
    "Validation PR-AUC:",
    average_precision_score(y_val, xgb_val_prob)
)

print(
    "Test ROC-AUC:",
    roc_auc_score(y_test, xgb_test_prob)
)

print(
    "Test PR-AUC:",
    average_precision_score(y_test, xgb_test_prob)
)

Scale positive weight: 2212.890909090909

XGBoost
Validation ROC-AUC: 0.7332739058984762
Validation PR-AUC: 0.0010880948813902836
Test ROC-AUC: 0.9612989323843416
Test PR-AUC: 0.5401983218916857


In [64]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    "Validation ROC-AUC": [
        roc_auc_score(y_val, log_val_prob),
        roc_auc_score(y_val, rf_val_prob),
        roc_auc_score(y_val, xgb_val_prob)
    ],

    "Validation PR-AUC": [
        average_precision_score(y_val, log_val_prob),
        average_precision_score(y_val, rf_val_prob),
        average_precision_score(y_val, xgb_val_prob)
    ],

    "Test ROC-AUC": [
        roc_auc_score(y_test, log_test_prob),
        roc_auc_score(y_test, rf_test_prob),
        roc_auc_score(y_test, xgb_test_prob)
    ],

    "Test PR-AUC": [
        average_precision_score(y_test, log_test_prob),
        average_precision_score(y_test, rf_test_prob),
        average_precision_score(y_test, xgb_test_prob)
    ]
})

display(comparison)

,Model,Validation ROC-AUC,Validation PR-AUC,Test ROC-AUC,Test PR-AUC
0,Logistic Regression,0.603983,0.000638,0.959242,0.098914
1,Random Forest,0.641659,0.000889,0.970349,0.176565
2,XGBoost,0.733274,0.001088,0.961299,0.540198


In [65]:
# ============================================================
# MODEL SPLIT AUDIT
# ============================================================

def split_summary(name, y, prob):
    print(f"\n{name}")
    print("-" * 50)

    print("Observations:", len(y))
    print("Failures:", int(y.sum()))
    print("Failure rate:", float(y.mean()))

    if y.sum() > 0:
        print(
            "PR-AUC:",
            average_precision_score(y, prob)
        )
        print(
            "ROC-AUC:",
            roc_auc_score(y, prob)
        )


split_summary(
    "VALIDATION",
    y_val,
    xgb_val_prob
)

split_summary(
    "TEST",
    y_test,
    xgb_test_prob
)


VALIDATION
--------------------------------------------------
Observations: 18332
Failures: 9
Failure rate: 0.0004909447959851625
PR-AUC: 0.0010880948813902836
ROC-AUC: 0.7332739058984762

TEST
--------------------------------------------------
Observations: 8996
Failures: 4
Failure rate: 0.00044464206313917296
PR-AUC: 0.5401983218916857
ROC-AUC: 0.9612989323843416


# Build the network

In [66]:
network_date = ml_data["Reporting Period End Date"].max()

print("Network date:", network_date)

Network date: 2025-06-30 00:00:00


In [67]:
snapshot = bank_quarter_panel[
    bank_quarter_panel["Reporting Period End Date"] == network_date
].copy()

snapshot = snapshot.dropna(
    subset=["IDRSSD", "total_assets"]
)

snapshot = (
    snapshot
    .sort_values("total_assets", ascending=False)
    .head(300)
    .copy()
)

network_banks = snapshot["IDRSSD"].astype(str).unique()

print("Selected banks:", len(network_banks))

display(
    snapshot[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "capital_ratio"
        ]
    ].head(10)
)

Selected banks: 300


,IDRSSD,Financial Institution Name,total_assets,capital_ratio
136739,852218,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",3788551000,0.085198
83100,480228,"BANK OF AMERICA, NATIONAL ASSOCIATION",2665555000,0.093992
82468,476810,"CITIBANK, N.A.",1833933000,0.096031
78737,451965,"WELLS FARGO BANK, NATIONAL ASSOCIATION",1746394000,0.09833
86750,504713,U.S. BANK NATIONAL ASSOCIATION,671438131,0.097376
5050,112837,"CAPITAL ONE, NATIONAL ASSOCIATION",648909392,0.149696
22676,2182786,GOLDMAN SACHS BANK USA,625410000,0.104856
131645,817824,"PNC BANK, NATIONAL ASSOCIATION",554573142,0.102155
136804,852320,TRUIST BANK,535585000,0.115321
92216,541101,"BANK OF NEW YORK MELLON, THE",398293000,0.077624


In [68]:
lookback_quarters = 20

all_dates = (
    bank_quarter_panel["Reporting Period End Date"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

history_dates = (
    all_dates[all_dates <= network_date]
    .tail(lookback_quarters)
)

print("Number of quarters:", len(history_dates))
print("Start:", history_dates.min())
print("End:", history_dates.max())

Number of quarters: 20
Start: 2020-09-30 00:00:00
End: 2025-06-30 00:00:00


In [69]:
network_features = [
    "capital_ratio",
    "loan_to_asset",
    "deposit_to_asset",
    "nonaccrual_to_asset",
    "nonaccrual_to_loan",
    "roa",
    "nii_to_asset"
]

print("Number of layers:", len(network_features))
print(network_features)

Number of layers: 7
['capital_ratio', 'loan_to_asset', 'deposit_to_asset', 'nonaccrual_to_asset', 'nonaccrual_to_loan', 'roa', 'nii_to_asset']


In [70]:
layer_correlations = {}
layer_edges = {}

network_history = bank_quarter_panel[
    bank_quarter_panel["IDRSSD"].astype(str).isin(network_banks)
    & bank_quarter_panel["Reporting Period End Date"].isin(history_dates)
].copy()

network_history["IDRSSD"] = (
    network_history["IDRSSD"].astype(str)
)

for feature in network_features:

    print(f"\nBuilding: {feature}")

    pivot = network_history.pivot(
        index="Reporting Period End Date",
        columns="IDRSSD",
        values=feature
    )

    pivot = pivot.reindex(
        columns=network_banks
    )

    corr = pivot.corr(
        method="pearson",
        min_periods=10
    )

    layer_correlations[feature] = corr

    threshold = 0.8

    edge_matrix = corr >= threshold

    # No self edges
    np.fill_diagonal(
        edge_matrix.values,
        False
    )

    rows, cols = np.where(
        np.triu(edge_matrix.values, k=1)
    )

    edges = {
        (corr.index[i], corr.columns[j])
        for i, j in zip(rows, cols)
    }

    layer_edges[feature] = edges

    print("Banks:", len(corr))
    print("Edges:", len(edges))


Building: capital_ratio
Banks: 300
Edges: 4851

Building: loan_to_asset
Banks: 300
Edges: 7532

Building: deposit_to_asset
Banks: 300
Edges: 1790

Building: nonaccrual_to_asset
Banks: 300
Edges: 2553

Building: nonaccrual_to_loan
Banks: 300
Edges: 2417

Building: roa
Banks: 300
Edges: 326

Building: nii_to_asset
Banks: 300
Edges: 5542


In [71]:
layer_summary = pd.DataFrame({
    "feature": network_features,
    "nodes": [
        len(layer_correlations[f])
        for f in network_features
    ],
    "edges": [
        len(layer_edges[f])
        for f in network_features
    ]
})

display(layer_summary)

,feature,nodes,edges
0,capital_ratio,300,4851
1,loan_to_asset,300,7532
2,deposit_to_asset,300,1790
3,nonaccrual_to_asset,300,2553
4,nonaccrual_to_loan,300,2417
5,roa,300,326
6,nii_to_asset,300,5542


In [72]:
final_edges = set(
    layer_edges[network_features[0]]
)

for feature in network_features[1:]:
    final_edges &= layer_edges[feature]

print("Final mT-MDS edges:", len(final_edges))

Final mT-MDS edges: 0


In [73]:
# ============================================================
# DEBUG THE MULTILAYER INTERSECTION
# ============================================================

print("Edges in each individual layer:\n")

for feature in network_features:
    print(
        f"{feature:25s}: "
        f"{len(layer_edges[feature]):6d} edges"
    )

print("\nProgressive intersection:\n")

current_edges = set(
    layer_edges[network_features[0]]
)

print(
    f"After {network_features[0]:25s}: "
    f"{len(current_edges):6d}"
)

for feature in network_features[1:]:

    current_edges &= layer_edges[feature]

    print(
        f"+ {feature:22s}: "
        f"{len(current_edges):6d}"
    )

Edges in each individual layer:

capital_ratio            :   4851 edges
loan_to_asset            :   7532 edges
deposit_to_asset         :   1790 edges
nonaccrual_to_asset      :   2553 edges
nonaccrual_to_loan       :   2417 edges
roa                      :    326 edges
nii_to_asset             :   5542 edges

Progressive intersection:

After capital_ratio            :   4851
+ loan_to_asset         :   1411
+ deposit_to_asset      :     76
+ nonaccrual_to_asset   :      7
+ nonaccrual_to_loan    :      5
+ roa                   :      0
+ nii_to_asset          :      0


In [74]:
# ============================================================
# INSPECT THE 5 EDGES THAT SURVIVE THE FIRST 5 LAYERS
# ============================================================

features_before_roa = [
    "capital_ratio",
    "loan_to_asset",
    "deposit_to_asset",
    "nonaccrual_to_asset",
    "nonaccrual_to_loan"
]

surviving_edges = set(
    layer_edges[features_before_roa[0]]
)

for feature in features_before_roa[1:]:
    surviving_edges &= layer_edges[feature]

print("Edges surviving first 5 layers:",
      len(surviving_edges))

diagnostic_rows = []

for bank_a, bank_b in surviving_edges:

    row = {
        "bank_a": bank_a,
        "bank_b": bank_b
    }

    for feature in network_features:
        row[feature] = layer_correlations[
            feature
        ].loc[bank_a, bank_b]

    diagnostic_rows.append(row)

surviving_diagnostics = pd.DataFrame(
    diagnostic_rows
)

display(
    surviving_diagnostics.sort_values(
        "roa",
        ascending=False
    )
)

Edges surviving first 5 layers: 5


,bank_a,bank_b,capital_ratio,loan_to_asset,deposit_to_asset,nonaccrual_to_asset,nonaccrual_to_loan,roa,nii_to_asset
2,63069,485559,0.878101,0.805625,0.862749,0.907600,0.855952,0.285827,0.812156
3,659855,871769,0.925566,0.824609,0.895225,0.902515,0.880998,0.260474,0.058854
4,940311,37640,0.854174,0.910208,0.902712,0.894698,0.925683,0.148727,0.563128
0,365325,702117,0.808384,0.967112,0.812706,0.801582,0.828004,0.036096,0.828782
1,940311,510871,0.914994,0.954213,0.806993,0.891867,0.908880,-0.244439,0.905375


In [75]:
# ============================================================
# 7-LAYER THRESHOLD SENSITIVITY
# ============================================================

thresholds = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

results_7layer = []

for threshold in thresholds:

    threshold_edges = {}

    for feature in network_features:

        corr = layer_correlations[feature]

        edge_matrix = corr >= threshold

        np.fill_diagonal(
            edge_matrix.values,
            False
        )

        rows, cols = np.where(
            np.triu(edge_matrix.values, k=1)
        )

        threshold_edges[feature] = {
            (corr.index[i], corr.columns[j])
            for i, j in zip(rows, cols)
        }

    final = set(
        threshold_edges[network_features[0]]
    )

    for feature in network_features[1:]:
        final &= threshold_edges[feature]

    results_7layer.append({
        "threshold": threshold,
        "final_edges": len(final)
    })

threshold_7_df = pd.DataFrame(results_7layer)

display(threshold_7_df)

,threshold,final_edges
0,0.60,13
1,0.65,1
2,0.70,0
3,0.75,0
4,0.80,0
5,0.85,0
6,0.90,0


In [76]:
network_specs = {
    "2_layers": [
        "capital_ratio",
        "roa"
    ],

    "3_layers": [
        "capital_ratio",
        "nonaccrual_to_asset",
        "roa"
    ],

    "5_layers": [
        "capital_ratio",
        "loan_to_asset",
        "deposit_to_asset",
        "nonaccrual_to_asset",
        "roa"
    ],

    "7_layers": [
        "capital_ratio",
        "loan_to_asset",
        "deposit_to_asset",
        "nonaccrual_to_asset",
        "nonaccrual_to_loan",
        "roa",
        "nii_to_asset"
    ]
}

In [77]:
# ============================================================
# COMPARE DIFFERENT MULTILAYER SPECIFICATIONS
# ============================================================

spec_results = []

threshold = 0.8

for spec_name, features in network_specs.items():

    final = set(
        layer_edges[features[0]]
    )

    for feature in features[1:]:
        final &= layer_edges[feature]

    spec_results.append({
        "specification": spec_name,
        "number_of_layers": len(features),
        "threshold": threshold,
        "final_edges": len(final)
    })

spec_df = pd.DataFrame(spec_results)

display(spec_df)

,specification,number_of_layers,threshold,final_edges
0,2_layers,2,0.8,42
1,3_layers,3,0.8,3
2,5_layers,5,0.8,0
3,7_layers,7,0.8,0


In [78]:
# ============================================================
# CORRELATION BETWEEN THE FINANCIAL FEATURES
# ============================================================

feature_corr = (
    network_history[network_features]
    .corr()
)

display(feature_corr.round(3))

,capital_ratio,loan_to_asset,deposit_to_asset,nonaccrual_to_asset,nonaccrual_to_loan,roa,nii_to_asset
capital_ratio,1.000,0.124,-0.427,0.282,0.255,0.205,0.244
loan_to_asset,0.124,1.000,0.005,0.097,-0.201,0.013,0.254
deposit_to_asset,-0.427,0.005,1.000,-0.102,-0.215,-0.133,-0.124
nonaccrual_to_asset,0.282,0.097,-0.102,1.000,0.783,0.040,0.015
nonaccrual_to_loan,0.255,-0.201,-0.215,0.783,1.000,0.120,-0.025
roa,0.205,0.013,-0.133,0.040,0.120,1.000,0.385
nii_to_asset,0.244,0.254,-0.124,0.015,-0.025,0.385,1.000


In [79]:
# ============================================================
# ADD PAPER-ALIGNED TIE FEATURE
# ============================================================

tie_frames = []

for year in years:

    path = FFIEC_DIR / str(year) / "part2.txt"

    print(f"Reading {year}...")

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[
            "IDRSSD",
            "Reporting Period End Date",
            "RIAD4073",
            "RIAD4107"
        ]
    )

    # Remove metadata row
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = (
        df["IDRSSD"]
        .astype("string")
        .str.strip()
    )

    df["Reporting Period End Date"] = pd.to_datetime(
        df["Reporting Period End Date"],
        errors="coerce"
    )

    # YTD values
    df["ytd_interest_expense"] = pd.to_numeric(
        df["RIAD4073"],
        errors="coerce"
    )

    df["ytd_interest_income"] = pd.to_numeric(
        df["RIAD4107"],
        errors="coerce"
    )

    # Year and quarter
    df["year"] = df["Reporting Period End Date"].dt.year
    df["quarter"] = df["Reporting Period End Date"].dt.quarter

    # Previous quarter within same bank/year
    previous_expense = (
        df.groupby(["IDRSSD", "year"])
        ["ytd_interest_expense"]
        .shift(1)
    )

    previous_income = (
        df.groupby(["IDRSSD", "year"])
        ["ytd_interest_income"]
        .shift(1)
    )

    previous_quarter = (
        df.groupby(["IDRSSD", "year"])
        ["quarter"]
        .shift(1)
    )

    # Convert YTD → quarterly
    df["quarterly_interest_expense"] = np.where(
        df["quarter"] == 1,
        df["ytd_interest_expense"],
        np.where(
            previous_quarter == df["quarter"] - 1,
            df["ytd_interest_expense"] - previous_expense,
            np.nan
        )
    )

    df["quarterly_interest_income"] = np.where(
        df["quarter"] == 1,
        df["ytd_interest_income"],
        np.where(
            previous_quarter == df["quarter"] - 1,
            df["ytd_interest_income"] - previous_income,
            np.nan
        )
    )

    # TIE ratio
    df["tie_ratio"] = np.where(
        df["quarterly_interest_income"] != 0,
        df["quarterly_interest_expense"]
        / df["quarterly_interest_income"],
        np.nan
    )

    tie_frames.append(
        df[
            [
                "IDRSSD",
                "Reporting Period End Date",
                "quarterly_interest_expense",
                "quarterly_interest_income",
                "tie_ratio"
            ]
        ]
    )

tie_panel = pd.concat(
    tie_frames,
    ignore_index=True
)

print("\nTIE panel:", tie_panel.shape)

Reading 2018...
Reading 2019...
Reading 2020...
Reading 2021...
Reading 2022...
Reading 2023...
Reading 2024...
Reading 2025...

TIE panel: (157921, 5)


In [80]:
# ============================================================
# MERGE TIE INTO MAIN PANEL
# ============================================================

bank_quarter_panel = bank_quarter_panel.merge(
    tie_panel,
    on=[
        "IDRSSD",
        "Reporting Period End Date"
    ],
    how="left",
    validate="one_to_one"
)

print(
    "bank_quarter_panel:",
    bank_quarter_panel.shape
)

print(
    "\nTIE missing:",
    bank_quarter_panel["tie_ratio"].isna().sum()
)

print(
    "TIE summary:"
)

print(
    bank_quarter_panel["tie_ratio"].describe()
)

bank_quarter_panel: (157921, 28)

TIE missing: 117970
TIE summary:
count    39951.000000
mean         0.195986
std          0.302819
min         -0.135324
25%          0.089310
50%          0.159851
75%          0.269049
max         28.545455
Name: tie_ratio, dtype: float64


In [81]:
# ============================================================
# TIE SANITY CHECK
# ============================================================

display(
    bank_quarter_panel[
        [
            "IDRSSD",
            "Reporting Period End Date",
            "quarterly_interest_expense",
            "quarterly_interest_income",
            "tie_ratio"
        ]
    ]
    .dropna(subset=["tie_ratio"])
    .head(20)
)

,IDRSSD,Reporting Period End Date,quarterly_interest_expense,quarterly_interest_income,tie_ratio
0,1000052,2018-03-31,307.0,3523.0,0.087142
4,1000052,2019-03-31,667.0,4275.0,0.156023
8,1000052,2020-03-31,669.0,4473.0,0.149564
12,1000052,2021-03-31,313.0,4017.0,0.077919
16,1000052,2022-03-31,214.0,4089.0,0.052336
20,1000052,2023-03-31,755.0,5465.0,0.138152
24,1000052,2024-03-31,2197.0,7282.0,0.301703
28,1000052,2025-03-31,2224.0,7932.0,0.280383
32,1000100,2018-03-31,12316.0,52364.0,0.235200
36,1000100,2019-03-31,22036.0,64591.0,0.341162


In [82]:
# ============================================================
# PAPER-ALIGNED 2-LAYER NETWORK
# ============================================================

network_features_paper = [
    "capital_ratio",
    "tie_ratio"
]

threshold = 0.8

layer_correlations_paper = {}
layer_edges_paper = {}

network_history_paper = bank_quarter_panel[
    bank_quarter_panel["IDRSSD"].astype(str).isin(network_banks)
    & bank_quarter_panel["Reporting Period End Date"].isin(history_dates)
].copy()

network_history_paper["IDRSSD"] = (
    network_history_paper["IDRSSD"].astype(str)
)

for feature in network_features_paper:

    print(f"\nBuilding layer: {feature}")

    pivot = network_history_paper.pivot(
        index="Reporting Period End Date",
        columns="IDRSSD",
        values=feature
    )

    pivot = pivot.reindex(
        columns=network_banks
    )

    corr = pivot.corr(
        method="pearson",
        min_periods=10
    )

    layer_correlations_paper[feature] = corr

    edge_matrix = corr >= threshold

    np.fill_diagonal(
        edge_matrix.values,
        False
    )

    rows, cols = np.where(
        np.triu(edge_matrix.values, k=1)
    )

    edges = {
        (corr.index[i], corr.columns[j])
        for i, j in zip(rows, cols)
    }

    layer_edges_paper[feature] = edges

    print(
        "Nodes:",
        len(corr)
    )

    print(
        "Edges:",
        len(edges)
    )


Building layer: capital_ratio
Nodes: 300
Edges: 4851

Building layer: tie_ratio
Nodes: 300
Edges: 0


In [86]:
# ============================================================
# FIXED TIE CONSTRUCTION
# IMPORTANT: SORT ASCENDING BEFORE YTD -> QUARTERLY
# ============================================================

tie_frames = []

for year in years:

    path = FFIEC_DIR / str(year) / "part2.txt"

    print(f"Processing {year}...")

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[
            "IDRSSD",
            "Reporting Period End Date",
            "RIAD4073",
            "RIAD4107"
        ]
    )

    # Remove metadata row
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = (
        df["IDRSSD"]
        .astype("string")
        .str.strip()
    )

    df["Reporting Period End Date"] = pd.to_datetime(
        df["Reporting Period End Date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # IMPORTANT FIX
    # Sort chronologically BEFORE calculating differences
    # --------------------------------------------------------

    df = df.sort_values(
        ["IDRSSD", "Reporting Period End Date"]
    ).reset_index(drop=True)

    # YTD values
    df["ytd_interest_expense"] = pd.to_numeric(
        df["RIAD4073"],
        errors="coerce"
    )

    df["ytd_interest_income"] = pd.to_numeric(
        df["RIAD4107"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Convert YTD -> quarterly
    # --------------------------------------------------------

    df = ytd_to_quarterly(
        df,
        [
            "ytd_interest_expense",
            "ytd_interest_income"
        ]
    )

    # --------------------------------------------------------
    # TIE = quarterly interest expense /
    #       quarterly interest income
    # --------------------------------------------------------

    df["tie_ratio"] = np.where(
        df["quarterly_interest_income"] != 0,
        df["quarterly_interest_expense"]
        / df["quarterly_interest_income"],
        np.nan
    )

    df["tie_ratio"] = (
        df["tie_ratio"]
        .replace([np.inf, -np.inf], np.nan)
    )

    tie_frames.append(
        df[
            [
                "IDRSSD",
                "Reporting Period End Date",
                "quarterly_interest_expense",
                "quarterly_interest_income",
                "tie_ratio"
            ]
        ]
    )

tie_panel_fixed = pd.concat(
    tie_frames,
    ignore_index=True
)

print("\nFixed TIE panel:", tie_panel_fixed.shape)

Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...

Fixed TIE panel: (157921, 5)


In [87]:
# ============================================================
# CHECK TIE COVERAGE AFTER FIX
# ============================================================

print(
    "Valid TIE observations:",
    tie_panel_fixed["tie_ratio"].notna().sum()
)

print(
    "Missing TIE observations:",
    tie_panel_fixed["tie_ratio"].isna().sum()
)

print("\nValid TIE observations by quarter:")

print(
    tie_panel_fixed
    .assign(
        quarter=
        tie_panel_fixed[
            "Reporting Period End Date"
        ].dt.quarter
    )
    .groupby("quarter")["tie_ratio"]
    .apply(lambda x: x.notna().sum())
)

Valid TIE observations: 157801
Missing TIE observations: 120

Valid TIE observations by quarter:
quarter
1    39951
2    39615
3    39310
4    38925
Name: tie_ratio, dtype: int64


In [88]:
# ============================================================
# CHECK ONE BANK
# ============================================================

display(
    tie_panel_fixed[
        tie_panel_fixed["IDRSSD"] == "1000052"
    ][
        [
            "IDRSSD",
            "Reporting Period End Date",
            "quarterly_interest_expense",
            "quarterly_interest_income",
            "tie_ratio"
        ]
    ].sort_values(
        "Reporting Period End Date"
    )
)

,IDRSSD,Reporting Period End Date,quarterly_interest_expense,quarterly_interest_income,tie_ratio
0,1000052,2018-03-31,307.0,3523.0,0.087142
1,1000052,2018-06-30,335.0,3654.0,0.091680
2,1000052,2018-09-30,377.0,3848.0,0.097973
3,1000052,2018-12-31,722.0,4402.0,0.164016
22233,1000052,2019-03-31,667.0,4275.0,0.156023
22234,1000052,2019-06-30,753.0,4559.0,0.165168
22235,1000052,2019-09-30,775.0,4617.0,0.167858
22236,1000052,2019-12-31,736.0,4562.0,0.161333
43531,1000052,2020-03-31,669.0,4473.0,0.149564
43532,1000052,2020-06-30,624.0,4445.0,0.140382


In [89]:
# ============================================================
# REPLACE OLD TIE COLUMNS
# ============================================================

old_tie_cols = [
    "quarterly_interest_expense",
    "quarterly_interest_income",
    "tie_ratio"
]

bank_quarter_panel = bank_quarter_panel.drop(
    columns=[
        c for c in old_tie_cols
        if c in bank_quarter_panel.columns
    ]
)

bank_quarter_panel = bank_quarter_panel.merge(
    tie_panel_fixed,
    on=[
        "IDRSSD",
        "Reporting Period End Date"
    ],
    how="left",
    validate="one_to_one"
)

print(
    "Updated bank_quarter_panel:",
    bank_quarter_panel.shape
)

print(
    "Valid TIE:",
    bank_quarter_panel["tie_ratio"].notna().sum()
)

Updated bank_quarter_panel: (157921, 28)
Valid TIE: 157801


In [90]:
# ============================================================
# REBUILD NETWORK HISTORY
# ============================================================

network_history_paper = bank_quarter_panel[
    bank_quarter_panel["IDRSSD"].astype(str).isin(
        network_banks
    )
    &
    bank_quarter_panel[
        "Reporting Period End Date"
    ].isin(history_dates)
].copy()

network_history_paper["IDRSSD"] = (
    network_history_paper["IDRSSD"]
    .astype(str)
)

print(
    "Network history shape:",
    network_history_paper.shape
)

print(
    "\nTIE observations:",
    network_history_paper["tie_ratio"].notna().sum()
)

Network history shape: (6000, 28)

TIE observations: 6000


In [91]:
# ============================================================
# REBUILD TIE CORRELATION
# ============================================================

tie_pivot = network_history_paper.pivot(
    index="Reporting Period End Date",
    columns="IDRSSD",
    values="tie_ratio"
)

tie_corr_fixed = tie_pivot.corr(
    method="pearson",
    min_periods=10
)

print(
    "Correlation matrix shape:",
    tie_corr_fixed.shape
)

# Unique pairwise correlations
mask = np.triu(
    np.ones(
        tie_corr_fixed.shape,
        dtype=bool
    ),
    k=1
)

pairwise_tie_corr = (
    tie_corr_fixed
    .where(mask)
    .stack()
)

print(
    "Number of valid bank pairs:",
    len(pairwise_tie_corr)
)

print(
    "Maximum correlation:",
    pairwise_tie_corr.max()
)

print(
    "\nCorrelation quantiles:"
)

display(
    pairwise_tie_corr.quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00
        ]
    )
)

Correlation matrix shape: (300, 300)
Number of valid bank pairs: 44850
Maximum correlation: 0.9992361790905787

Correlation quantiles:


0.50    0.972565
0.75    0.986992
0.90    0.993000
0.95    0.995049
0.99    0.997375
1.00    0.999236
dtype: float64

In [92]:
# ============================================================
# CORRECTED TIE LAYER @ 0.8
# ============================================================

threshold = 0.8

tie_edge_matrix = (
    tie_corr_fixed >= threshold
)

np.fill_diagonal(
    tie_edge_matrix.values,
    False
)

rows, cols = np.where(
    np.triu(
        tie_edge_matrix.values,
        k=1
    )
)

tie_edges_fixed = {
    (
        tie_corr_fixed.index[i],
        tie_corr_fixed.columns[j]
    )
    for i, j in zip(rows, cols)
}

print(
    "TIE edges at threshold 0.8:",
    len(tie_edges_fixed)
)

TIE edges at threshold 0.8: 41687


In [93]:
# ============================================================
# CORRECTED 2-LAYER mT-MDS
# ============================================================

capital_edges = layer_edges_paper[
    "capital_ratio"
]

final_edges_paper_fixed = (
    capital_edges
    & tie_edges_fixed
)

print(
    "Capital edges:",
    len(capital_edges)
)

print(
    "TIE edges:",
    len(tie_edges_fixed)
)

print(
    "Final mT-MDS edges:",
    len(final_edges_paper_fixed)
)

Capital edges: 4851
TIE edges: 41687
Final mT-MDS edges: 2308


In [95]:
!pip install "networkx[default]"


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [96]:

# ============================================================
# FINAL 2-LAYER mT-MDS GRAPH
# ============================================================
import networkx as nx
G_mtds = nx.Graph()

# Add all selected banks, including isolated ones
G_mtds.add_nodes_from(network_banks)

# Add edges that survived both layers
G_mtds.add_edges_from(final_edges_paper_fixed)

print("Number of nodes:", G_mtds.number_of_nodes())
print("Number of edges:", G_mtds.number_of_edges())

isolated_nodes = list(nx.isolates(G_mtds))

print("Isolated nodes:", len(isolated_nodes))

print(
    "Interconnected nodes:",
    G_mtds.number_of_nodes() - len(isolated_nodes)
)

Number of nodes: 300
Number of edges: 2308
Isolated nodes: 69
Interconnected nodes: 231


In [97]:
# ============================================================
# CONNECTED COMPONENTS
# ============================================================

components = list(
    nx.connected_components(G_mtds)
)

component_sizes = sorted(
    [len(component) for component in components],
    reverse=True
)

print("Connected components:",
      len(components))

print("Largest component:",
      component_sizes[0])

print("Largest 10 components:",
      component_sizes[:10])

Connected components: 73
Largest component: 225
Largest 10 components: [225, 2, 2, 2, 1, 1, 1, 1, 1, 1]


In [98]:
# ============================================================
# NODE DEGREE
# ============================================================

degree_dict = dict(
    G_mtds.degree()
)

degree_table = snapshot[
    [
        "IDRSSD",
        "Financial Institution Name",
        "total_assets",
        "capital_ratio"
    ]
].copy()

degree_table["IDRSSD"] = (
    degree_table["IDRSSD"]
    .astype(str)
)

degree_table["degree"] = (
    degree_table["IDRSSD"]
    .map(degree_dict)
)

degree_table = degree_table.sort_values(
    "degree",
    ascending=False
)

display(
    degree_table.head(20)
)

,IDRSSD,Financial Institution Name,total_assets,capital_ratio,degree
117614,717924,SOUTHERN BANK AND TRUST COMPANY,4943138,0.079273,69
132939,825146,MORTON COMMUNITY BANK,5468022,0.082955,67
24957,233031,REGIONS BANK,157807000,0.111193,65
37784,280110,KEYBANK NATIONAL ASSOCIATION,182591692,0.100503,64
36433,276579,"ZIONS BANCORPORATION, NATIONAL ASSOCIATION",88892748,0.074199,63
106036,636771,INDEPENDENT BANK,5411119,0.091001,58
142255,888002,MIDDLESEX SAVINGS BANK,6102056,0.120974,58
20731,208244,OLD NATIONAL BANK,70701190,0.111341,58
70474,395238,SYNOVUS BANK,60919986,0.091962,56
40807,288853,FIRSTBANK,26761023,0.063432,54


In [99]:
# ============================================================
# GREEDY DOMINATING SET
# ============================================================

greedy_mds = nx.dominating_set(
    G_mtds
)

print("Greedy dominating-set size:",
      len(greedy_mds))

print("Total nodes:",
      G_mtds.number_of_nodes())

print(
    "Fraction of nodes selected:",
    len(greedy_mds)
    / G_mtds.number_of_nodes()
)

Greedy dominating-set size: 136
Total nodes: 300
Fraction of nodes selected: 0.4533333333333333


In [100]:
# ============================================================
# VERIFY DOMINATION
# ============================================================

uncovered = []

for node in G_mtds.nodes():

    # Node itself is selected
    if node in greedy_mds:
        continue

    # Otherwise at least one neighbor must be selected
    if not any(
        neighbor in greedy_mds
        for neighbor in G_mtds.neighbors(node)
    ):
        uncovered.append(node)

print("Uncovered nodes:", len(uncovered))

Uncovered nodes: 0


In [101]:
isolated_in_mds = (
    set(isolated_nodes)
    & set(greedy_mds)
)

print(
    "Isolated banks:",
    len(isolated_nodes)
)

print(
    "Isolated banks included in MDS:",
    len(isolated_in_mds)
)

print(
    "All isolated nodes included:",
    set(isolated_nodes)
    == isolated_in_mds
)

Isolated banks: 69
Isolated banks included in MDS: 69
All isolated nodes included: True


In [102]:
# ============================================================
# MDS BANKS
# ============================================================

mds_banks = degree_table[
    degree_table["IDRSSD"].isin(
        set(greedy_mds)
    )
].copy()

mds_banks = mds_banks.sort_values(
    "degree",
    ascending=False
)

print(
    "Number of MDS banks:",
    len(mds_banks)
)

display(mds_banks)

Number of MDS banks: 136


,IDRSSD,Financial Institution Name,total_assets,capital_ratio,degree
81252,470050,FIRST FINANCIAL BANK,14324488,0.111948,49
74158,419406,BANGOR SAVINGS BANK,7335916,0.063747,39
100264,595869,CATHAY BANK,23709143,0.125227,31
110271,663245,SIMMONS BANK,26628111,0.134331,29
113102,682563,FROST BANK,51489892,0.079495,27
...,...,...,...,...,...
56721,3394278,SALLIE MAE BANK,29548460,0.099475,0
148802,934329,"BNY MELLON, NATIONAL ASSOCIATION",29578000,0.123943,0
153018,962966,"SOFI BANK, NATIONAL ASSOCIATION",36284122,0.135298,0
34905,2735146,"EVERBANK, NATIONAL ASSOCIATION",44446323,0.085605,0


In [103]:
# ============================================================
# MDS COVERAGE
# ============================================================

mds_coverage = []

for node in greedy_mds:

    covered_nodes = (
        {node}
        | set(G_mtds.neighbors(node))
    )

    mds_coverage.append({
        "IDRSSD": node,
        "degree": G_mtds.degree(node),
        "covered_nodes": len(covered_nodes)
    })

mds_coverage = pd.DataFrame(
    mds_coverage
)

mds_coverage = (
    mds_coverage
    .sort_values(
        "covered_nodes",
        ascending=False
    )
    .reset_index(drop=True)
)

display(mds_coverage.head(20))

,IDRSSD,degree,covered_nodes
0,470050,49,50
1,419406,39,40
2,595869,31,32
3,663245,29,30
4,682563,27,28
5,601416,27,28
6,436159,20,21
7,661308,20,21
8,693224,18,19
9,723112,18,19


In [104]:
# ============================================================
# READABLE MDS COVERAGE TABLE
# ============================================================

mds_coverage_readable = mds_coverage.merge(
    snapshot[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "capital_ratio"
        ]
    ].copy(),
    on="IDRSSD",
    how="left"
)

mds_coverage_readable = (
    mds_coverage_readable[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "capital_ratio",
            "degree",
            "covered_nodes"
        ]
    ]
)

display(
    mds_coverage_readable.head(20)
)

,IDRSSD,Financial Institution Name,total_assets,capital_ratio,degree,covered_nodes
0,470050,FIRST FINANCIAL BANK,14324488,0.111948,49,50
1,419406,BANGOR SAVINGS BANK,7335916,0.063747,39,40
2,595869,CATHAY BANK,23709143,0.125227,31,32
3,663245,SIMMONS BANK,26628111,0.134331,29,30
4,682563,FROST BANK,51489892,0.079495,27,28
5,601416,FIVE STAR BANK,6105894,0.102658,27,28
6,436159,FIRSTBANK,13323769,0.118005,20,21
7,661308,AMALGAMATED BANK,8623488,0.091986,20,21
8,693224,ATLANTIC UNION BANK,37203504,0.147146,18,19
9,723112,"FIFTH THIRD BANK, NATIONAL ASSOCIATION",209311000,0.112942,18,19


In [105]:
# ============================================================
# ASSIGN EACH BANK TO AN MDS REPRESENTATIVE
# ============================================================

mds_set = set(greedy_mds)

assignments = []

for node in G_mtds.nodes():

    if node in mds_set:
        representative = node

    else:
        mds_neighbors = [
            neighbor
            for neighbor in G_mtds.neighbors(node)
            if neighbor in mds_set
        ]

        if mds_neighbors:
            # For now, choose the MDS neighbor
            # with the largest degree
            representative = max(
                mds_neighbors,
                key=lambda x: G_mtds.degree(x)
            )
        else:
            representative = None

    assignments.append({
        "IDRSSD": node,
        "MDS_representative": representative
    })

assignment_df = pd.DataFrame(assignments)

print(
    "Banks assigned to an MDS representative:",
    assignment_df["MDS_representative"].notna().sum()
)

print(
    "Unassigned:",
    assignment_df["MDS_representative"].isna().sum()
)

Banks assigned to an MDS representative: 300
Unassigned: 0


In [106]:
# ============================================================
# NEIGHBORHOOD SIZES
# ============================================================

neighborhood_sizes = (
    assignment_df
    .groupby("MDS_representative")
    .size()
    .reset_index(
        name="neighborhood_size"
    )
    .sort_values(
        "neighborhood_size",
        ascending=False
    )
)

display(
    neighborhood_sizes.head(20)
)

,MDS_representative,neighborhood_size
85,470050,50
74,3579589,13
97,601416,10
102,661308,10
80,419406,9
15,169653,7
103,663245,7
82,436159,6
96,595869,6
99,633378,6


In [107]:
import scipy
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

print("SciPy version:", scipy.__version__)

SciPy version: 1.16.3


In [108]:
# ============================================================
# EXACT MINIMUM DOMINATING SET USING MILP
# ============================================================

def exact_mds_component(G_component):
    """
    Solve the exact Minimum Dominating Set problem
    for one connected component.

    min sum(x_i)

    subject to:
        x_i + sum(x_j for j in N(i)) >= 1
        x_i in {0,1}
    """

    nodes = list(G_component.nodes())
    n = len(nodes)

    node_to_idx = {
        node: i
        for i, node in enumerate(nodes)
    }

    # Constraint matrix:
    #
    # A[i, j] = 1 if j is node i itself
    #           or j is a neighbor of i
    #
    # This represents the closed neighborhood N[i].

    A = lil_matrix((n, n), dtype=float)

    for node in nodes:

        i = node_to_idx[node]

        # Self
        A[i, i] = 1.0

        # Neighbors
        for neighbor in G_component.neighbors(node):
            j = node_to_idx[neighbor]
            A[i, j] = 1.0

    A = A.tocsr()

    # Objective: minimize number of selected nodes
    c = np.ones(n)

    # Binary variables
    integrality = np.ones(n)

    bounds = Bounds(
        np.zeros(n),
        np.ones(n)
    )

    # Every node must be dominated:
    #
    # A x >= 1
    #
    constraints = LinearConstraint(
        A,
        np.ones(n),
        np.full(n, np.inf)
    )

    result = milp(
        c=c,
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options={
            "presolve": True
        }
    )

    if not result.success:
        raise RuntimeError(
            f"MILP failed: {result.message}"
        )

    selected = {
        nodes[i]
        for i, value in enumerate(result.x)
        if value > 0.5
    }

    return selected, result


# ------------------------------------------------------------
# Solve each connected component separately
# ------------------------------------------------------------

exact_mds = set()

component_results = []

components = list(
    nx.connected_components(G_mtds)
)

for component_nodes in sorted(
    components,
    key=len,
    reverse=True
):

    H = G_mtds.subgraph(component_nodes).copy()

    size = H.number_of_nodes()

    print(
        f"Solving component with {size} nodes..."
    )

    selected, result = exact_mds_component(H)

    exact_mds.update(selected)

    component_results.append({
        "component_size": size,
        "mds_size": len(selected),
        "objective": result.fun,
        "status": result.message
    })

exact_component_df = pd.DataFrame(
    component_results
)

print("\n================================")
print("EXACT MDS COMPLETE")
print("================================")

print(
    "Exact MDS size:",
    len(exact_mds)
)

print(
    "Greedy MDS size:",
    len(greedy_mds)
)

print(
    "Greedy excess:",
    len(greedy_mds) - len(exact_mds)
)

display(exact_component_df)

Solving component with 225 nodes...
Solving component with 2 nodes...
Solving component with 2 nodes...
Solving component with 2 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving component with 1 nodes...
Solving comp

,component_size,mds_size,objective,status
0,225,30,30.0,Optimization terminated successfully. (HiGHS S...
1,2,1,1.0,Optimization terminated successfully. (HiGHS S...
2,2,1,1.0,Optimization terminated successfully. (HiGHS S...
3,2,1,1.0,Optimization terminated successfully. (HiGHS S...
4,1,1,1.0,Optimization terminated successfully. (HiGHS S...
...,...,...,...,...
68,1,1,1.0,Optimization terminated successfully. (HiGHS S...
69,1,1,1.0,Optimization terminated successfully. (HiGHS S...
70,1,1,1.0,Optimization terminated successfully. (HiGHS S...
71,1,1,1.0,Optimization terminated successfully. (HiGHS S...


In [109]:
# ============================================================
# VERIFY EXACT MDS
# ============================================================

uncovered_exact = []

for node in G_mtds.nodes():

    if node in exact_mds:
        continue

    if not any(
        neighbor in exact_mds
        for neighbor in G_mtds.neighbors(node)
    ):
        uncovered_exact.append(node)

print(
    "Exact MDS size:",
    len(exact_mds)
)

print(
    "Uncovered nodes:",
    len(uncovered_exact)
)

print(
    "All nodes dominated:",
    len(uncovered_exact) == 0
)

Exact MDS size: 102
Uncovered nodes: 0
All nodes dominated: True


In [110]:
# ============================================================
# ISOLATED-NODE CHECK
# ============================================================

isolated_set = set(isolated_nodes)

missing_isolated = (
    isolated_set - exact_mds
)

print(
    "Total isolated:",
    len(isolated_set)
)

print(
    "Isolated nodes in exact MDS:",
    len(isolated_set & exact_mds)
)

print(
    "Missing isolated nodes:",
    len(missing_isolated)
)

Total isolated: 69
Isolated nodes in exact MDS: 69
Missing isolated nodes: 0


In [111]:
# ============================================================
# GREEDY VS EXACT
# ============================================================

mds_comparison = pd.DataFrame({
    "method": [
        "Greedy",
        "Exact MILP"
    ],
    "MDS_size": [
        len(greedy_mds),
        len(exact_mds)
    ],
    "fraction_of_network": [
        len(greedy_mds) / G_mtds.number_of_nodes(),
        len(exact_mds) / G_mtds.number_of_nodes()
    ]
})

display(mds_comparison)

,method,MDS_size,fraction_of_network
0,Greedy,136,0.453333
1,Exact MILP,102,0.340000


In [112]:
# ============================================================
# EXACT MDS BANK TABLE
# ============================================================

degree_dict = dict(
    G_mtds.degree()
)

exact_mds_banks = snapshot[
    snapshot["IDRSSD"]
    .astype(str)
    .isin(exact_mds)
].copy()

exact_mds_banks["degree"] = (
    exact_mds_banks["IDRSSD"]
    .astype(str)
    .map(degree_dict)
)

exact_mds_banks = exact_mds_banks.sort_values(
    "degree",
    ascending=False
)

print(
    "Exact MDS banks:",
    len(exact_mds_banks)
)

display(
    exact_mds_banks[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "capital_ratio",
            "degree"
        ]
    ]
)

Exact MDS banks: 102


,IDRSSD,Financial Institution Name,total_assets,capital_ratio,degree
24957,233031,REGIONS BANK,157807000,0.111193,65
115261,701062,CENTRAL PACIFIC BANK,7363041,0.0883,53
112962,680813,"FARMERS NATIONAL BANK OF CANFIELD, THE",5162387,0.092402,53
131549,816603,"WASHINGTON TRUST COMPANY, OF WESTERLY, THE",6744900,0.080696,45
48594,3150447,"CHARLES SCHWAB BANK, SSB",247664000,0.085749,41
...,...,...,...,...,...
39369,2850722,BMW BANK OF NORTH AMERICA,12280786,0.1416,0
106152,637451,PLAINSCAPITAL BANK,12409286,0.12104,0
50756,3224580,COMENITY CAPITAL BANK,12908706,0.146119,0
147828,929352,BELL BANK,14066290,0.099189,0


In [113]:
# ============================================================
# EXACT MDS NEIGHBORHOODS
# ============================================================

exact_mds_assignments = []

for node in G_mtds.nodes():

    if node in exact_mds:

        representative = node

    else:

        representatives = [
            neighbor
            for neighbor in G_mtds.neighbors(node)
            if neighbor in exact_mds
        ]

        if representatives:

            # If multiple exact-MDS nodes dominate this bank,
            # choose the one with the highest degree for display.
            representative = max(
                representatives,
                key=lambda x: G_mtds.degree(x)
            )

        else:
            representative = None

    exact_mds_assignments.append({
        "IDRSSD": node,
        "MDS_representative": representative
    })

exact_assignment_df = pd.DataFrame(
    exact_mds_assignments
)

print(
    "Assigned:",
    exact_assignment_df[
        "MDS_representative"
    ].notna().sum()
)

print(
    "Unassigned:",
    exact_assignment_df[
        "MDS_representative"
    ].isna().sum()
)

Assigned: 300
Unassigned: 0


In [114]:
# ============================================================
# EXACT MDS NEIGHBORHOOD SIZES
# ============================================================

exact_neighborhood_sizes = (
    exact_assignment_df
    .groupby("MDS_representative")
    .size()
    .reset_index(
        name="neighborhood_size"
    )
    .sort_values(
        "neighborhood_size",
        ascending=False
    )
    .reset_index(drop=True)
)

# Add bank information
exact_neighborhood_sizes = (
    exact_neighborhood_sizes
    .merge(
        snapshot[
            [
                "IDRSSD",
                "Financial Institution Name",
                "total_assets"
            ]
        ],
        left_on="MDS_representative",
        right_on="IDRSSD",
        how="left"
    )
)

display(
    exact_neighborhood_sizes.head(20)
)

,MDS_representative,neighborhood_size,IDRSSD,Financial Institution Name,total_assets
0,233031,63,233031,REGIONS BANK,157807000
1,677176,22,677176,TRUSTCO BANK,6347651
2,701062,17,701062,CENTRAL PACIFIC BANK,7363041
3,3150447,15,3150447,"CHARLES SCHWAB BANK, SSB",247664000
4,881478,13,881478,HERITAGE BANK,7065598
5,680813,13,680813,"FARMERS NATIONAL BANK OF CANFIELD, THE",5162387
6,816603,12,816603,"WASHINGTON TRUST COMPANY, OF WESTERLY, THE",6744900
7,782306,7,782306,BROOKLINE BANK,6162093
8,3284397,6,3284397,BEAL BANK USA,16985384
9,216922,4,216922,FIRST BANK,12602725
